# Research Evidence Agent
## Small Agentic AI Demonstration

- One LLM decision component
- Four deterministic tools
- An explicit observe-decide-act loop
- A deterministic provenance gate


## 1. Environment check

CPU execution is supported but may be slow for the default model.

In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No CUDA GPU detected; inference may be slow.")


## 2. Install the project and model dependencies

Run this notebook from the cloned repository directory. If the repository is private, either upload its ZIP to Colab and extract it, or clone it manually using an authenticated GitHub session/token. Do not paste or save a token in this notebook.


In [ ]:
# If needed, change this to the extracted/cloned repository directory first:
# %cd /content/research-evidence-agent
%pip install -q -e ".[colab]"


## 3. Load the decision model

The initial open-weight model download can take time.

In [ ]:
from research_agent.transformers_model import TransformersDecisionModel

model = TransformersDecisionModel(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=256,
)


## 4. Create the existing agent runtime

In [ ]:
from research_agent.agent import ResearchAgent
from research_agent.tools.registry import create_default_registry

registry = create_default_registry()
agent = ResearchAgent(model=model, registry=registry, max_steps=6)


## 5. Observable trace

This prints actions and tool outcomes only—never hidden model reasoning.

In [ ]:
import json

def print_run(result):
    print("STATUS:", result.status)
    print("QUESTION:", result.state.question)
    for observation in result.state.observations:
        print(f"\nSTEP: {observation.step}")
        print("TOOL:", observation.tool)
        print("ARGUMENTS:", json.dumps(observation.arguments, sort_keys=True))
        if observation.error is not None:
            print("ERROR:", observation.error)
        else:
            print("RESULT:", json.dumps(observation.result, indent=2, sort_keys=True))
    print("\nFINAL ANSWER:", result.answer)
    print("EVIDENCE IDS:", result.evidence_ids)


## 6. Run demo questions

The real model chooses each action; answers are not hard-coded.

In [ ]:
questions = [
    "What F1 score did LoRA Small achieve?",
    ("Which experiment should I choose if I need F1 of at least 0.72, "
     "latency below 300 ms, and local inference?"),
    "Which experiment used the least energy?",
]

for question in questions:
    print("\n" + "=" * 72)
    print_run(agent.run(question))


## 7. Deterministic fallback demonstration

Use this only if the GPU is unavailable, download fails, or inference is too slow. This is scripted behavior, **not** real LLM output. The `ResearchAgent`, registry, tools, state, and provenance gate are unchanged; only the `DecisionModel` is swapped.


In [ ]:
from research_agent.actions import FinalAction, ToolAction
from research_agent.model import ScriptedModel

def run_scripted_fallback():
    print("Deterministic fallback demonstration")
    scripted_model = ScriptedModel([
        ToolAction("search_notes", {"query": "LoRA Small F1"}),
        ToolAction("read_note", {"document_id": "lora-small"}),
        FinalAction("LoRA Small achieved an F1 score of 0.74.", ["lora-small"]),
    ])
    fallback_agent = ResearchAgent(
        model=scripted_model,
        registry=create_default_registry(),
        max_steps=6,
    )
    print_run(fallback_agent.run("What F1 score did LoRA Small achieve?"))

# run_scripted_fallback()
